In [1]:
import pandas as pd
from itertools import product

excel_file = "resultados.xlsx"

architectures = [
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]

r_values = [0.01, 0.5, 0.9]

N_MODELS = 5  # número de seeds

df = pd.read_excel(excel_file)

comb = list(product(architectures, r_values))

for i, (arch, r) in enumerate(comb):
    
    start = i * N_MODELS
    end = start + N_MODELS
    
    df.loc[start:end-1, "Neurons"] = str(arch)
    df.loc[start:end-1, "r"] = r

df.to_excel("resultados.xlsx", index=False)

C:\Users\Sarah\AppData\Local\Temp\ipykernel_11056\3340439140.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[32]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[start:end-1, "Neurons"] = str(arch)


In [2]:
import numpy as np

TARGETS = ["X"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [3]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [4]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,X,Train_1,model_4_0,"[8, 4]",[0.9804187802566534]
1,X,Train_2,model_8_2,"[16, 8]",[0.9908274588503753]
2,X,Val,model_16_3,"[32, 16]",[0.9003962983739532]
3,X,Test_1,model_32_3,"[64, 32]",[0.9847074070296642]
4,X,Test_2,model_64_0,"[128, 64]",[0.9929727168976836]
5,X,Test_3,model_128_2,[128],[0.6419393096761888]


In [5]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,X,0.965426,0.006905,0.003728,0.000745,0.980419,"[8, 4]"
1,Train_2,X,0.981990,0.006788,0.001518,0.000572,0.990827,"[16, 8]"
2,Val,X,0.881102,0.017154,0.011369,0.001640,0.900396,"[32, 16]"
3,Test_1,X,0.971977,0.005967,0.002936,0.000625,0.984707,"[64, 32]"
4,Test_2,X,0.987633,0.003724,0.001018,0.000306,0.992973,"[128, 64]"
5,Test_3,X,-1.712456,4.117976,0.249558,0.378872,0.641939,[128]


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted